
# Buy Fee Model V8 — Matched Control + Difference-in-Differences

## Objective

V7 showed:

- **RPU:** strongest signal
- **Conversion:** some directional signal, but weak magnitude prediction
- **Volume:** weak direct predictive signal

The key remaining problem is that simple pre/post comparisons cannot separate a fee effect from:

- seasonality,
- market-wide changes,
- location-wide changes,
- vehicle-mix shifts,
- common calendar shocks.

V8 addresses this using **matched controls and Difference-in-Differences (DiD)**.

---

## Core idea

For every treated segment:

**Location × Lane × Price Tier × Fee Change Date**

we find similar segments that:

1. Did **not** change Buy Fee in the same event window.
2. Have enough 8-week pre/post history.
3. Have similar pre-period:
   - registered volume,
   - sold volume,
   - conversion,
   - Buy Fee RPU,
   - sale amount,
   - vehicle age,
   - mileage,
   - condition,
   - floor/high bid,
   - trend/volatility.
4. Preferably come from **other locations** so the control is less exposed to the same local operational shock.

Then:

\[
DiD = (Treated_{Post}-Treated_{Pre}) - (Control_{Post}-Control_{Pre})
\]

This gives us an estimate of the **incremental change associated with the fee increase relative to what happened in comparable untreated segments at the same time**.

---

## Outputs

The notebook produces:

- matched treated/control pairs,
- match-quality diagnostics,
- DiD event-level results,
- +$10 / +$15 / +$20 / +$25 / +$30 scenario evidence,
- fixed-effects DiD regressions,
- grouped validation by `Corporate_Change_ID`,
- RPU fee pass-through after control adjustment,
- readiness assessment for:
  - Volume
  - Conversion
  - RPU


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
import gc

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, r2_score

import statsmodels.formula.api as smf

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V61_DIR = DATA_FOLDER / "buy_fee_model_dataset_v6_1"

WEEKLY_FILE = V61_DIR / "04_WEEKLY_MODEL_PANEL_V6_1.csv"
EVENTS_FILE = V61_DIR / "05_CLEAN_FEE_CHANGE_EVENTS_V6_1.csv"
STRICT_FILE = V61_DIR / "09_MODEL_READY_STRICT_V6_1.csv"
RPU_FILE = V61_DIR / "10_MODEL_READY_RPU_V6_1.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v8_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_WINDOW = 8
MIN_OBSERVED_WEEKS = 6
MIN_REGISTERED_PRE = 25
MIN_REGISTERED_POST = 25

# Matching
N_CONTROLS = 3
MAX_MATCH_DISTANCE = 4.0
PRICE_TIER_TOLERANCE = 3000   # +/- $3k around tier low
EXCLUDE_SAME_LOCATION = True

# Control contamination rule
CONTROL_CHANGE_EXCLUSION_WEEKS = 8

SCENARIOS = [10, 15, 20, 25, 30]

# Readiness thresholds
MIN_MATCH_RATE_GO = 0.70
MIN_UNIQUE_DATES_GO = 10
MIN_UNIQUE_LOCATIONS_GO = 10
MIN_SIGN_STABILITY_GO = 0.70

print("=" * 80)
print("V8 INPUT CHECK")
print("=" * 80)

for p in [WEEKLY_FILE, EVENTS_FILE, STRICT_FILE, RPU_FILE]:
    print(f"{p.name:<40} {p.exists()}")

missing = [p for p in [WEEKLY_FILE, EVENTS_FILE, STRICT_FILE, RPU_FILE] if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing V6.1 output(s):\n"
        + "\n".join(str(x) for x in missing)
    )


V8 INPUT CHECK
04_WEEKLY_MODEL_PANEL_V6_1.csv           True
05_CLEAN_FEE_CHANGE_EVENTS_V6_1.csv      True
09_MODEL_READY_STRICT_V6_1.csv           True
10_MODEL_READY_RPU_V6_1.csv              True


## 1. Load cleaned V6.1 data

In [3]:
# ============================================================
# MEMORY-SAFE LOAD OF V6.1 OUTPUTS
# ============================================================

print("=" * 80)
print("LOADING V6.1 DATA - MEMORY SAFE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Small files can be loaded normally
# ------------------------------------------------------------

events = pd.read_csv(
    EVENTS_FILE,
    parse_dates=[
        "BUYFEE_CHANGE_DATE",
        "Prev_Change_Date",
        "Next_Change_Date"
    ],
    low_memory=True
)

strict = pd.read_csv(
    STRICT_FILE,
    parse_dates=[
        "BUYFEE_CHANGE_DATE",
        "Event_Week"
    ],
    low_memory=True
)

rpu_ready = pd.read_csv(
    RPU_FILE,
    parse_dates=[
        "BUYFEE_CHANGE_DATE",
        "Event_Week"
    ],
    low_memory=True
)

print("Events loaded :", f"{len(events):,}")
print("Strict loaded :", f"{len(strict):,}")
print("RPU loaded    :", f"{len(rpu_ready):,}")


# ------------------------------------------------------------
# 2. Determine only the weekly date range actually needed
# ------------------------------------------------------------

strict["BUYFEE_CHANGE_DATE"] = pd.to_datetime(
    strict["BUYFEE_CHANGE_DATE"],
    errors="coerce"
)

min_change_date = strict[
    "BUYFEE_CHANGE_DATE"
].min()

max_change_date = strict[
    "BUYFEE_CHANGE_DATE"
].max()


# We need:
# 8 weeks pre
# event week
# 8 weeks post
# plus some buffer for safety

MIN_WEEK_NEEDED = (
    min_change_date
    - pd.Timedelta(weeks=PRIMARY_WINDOW + 2)
)

MAX_WEEK_NEEDED = (
    max_change_date
    + pd.Timedelta(weeks=PRIMARY_WINDOW + 2)
)

print("\nWeekly panel date range needed:")
print("From:", MIN_WEEK_NEEDED)
print("To  :", MAX_WEEK_NEEDED)


# ------------------------------------------------------------
# 3. Determine segments actually relevant to V8
# ------------------------------------------------------------

relevant_segments = (
    strict[
        [
            "VRLOCATION",
            "LANE",
            "price_tier"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print(
    "\nStrict treated segments:",
    f"{len(relevant_segments):,}"
)


# ------------------------------------------------------------
# 4. Memory-safe weekly loader
# ------------------------------------------------------------

WEEKLY_CHUNK_SIZE = 150_000

weekly_parts = []

weekly_rows_read = 0
weekly_rows_kept = 0

reader = pd.read_csv(
    WEEKLY_FILE,
    chunksize=WEEKLY_CHUNK_SIZE,
    low_memory=True,
    encoding="utf-8",
    encoding_errors="replace"
)


for chunk_num, chunk in enumerate(
    reader,
    start=1
):

    weekly_rows_read += len(chunk)

    # --------------------------------------------
    # Parse Week_Start only for this chunk
    # --------------------------------------------

    chunk["Week_Start"] = pd.to_datetime(
        chunk["Week_Start"],
        errors="coerce"
    )


    # --------------------------------------------
    # Keep only date range needed
    # --------------------------------------------

    chunk = chunk[
        chunk["Week_Start"].between(
            MIN_WEEK_NEEDED,
            MAX_WEEK_NEEDED
        )
    ].copy()


    if chunk.empty:

        if chunk_num % 5 == 0:

            print(
                f"Processed {chunk_num:,} chunks | "
                f"{weekly_rows_read:,} rows read | "
                f"{weekly_rows_kept:,} kept"
            )

        continue


    # --------------------------------------------
    # Normalize join fields
    # --------------------------------------------

    chunk["VRLOCATION"] = (
        chunk["VRLOCATION"]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )

    chunk["LANE"] = (
        chunk["LANE"]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )

    chunk["price_tier"] = (
        chunk["price_tier"]
        .astype("string")
        .fillna("")
        .str.strip()
    )


    # --------------------------------------------
    # IMPORTANT:
    # Keep all segments for matching controls
    #
    # So DO NOT filter only to treated segments.
    #
    # We only restrict by calendar range.
    # --------------------------------------------

    weekly_parts.append(
        chunk
    )

    weekly_rows_kept += len(chunk)


    if chunk_num % 5 == 0:

        print(
            f"Processed {chunk_num:,} chunks | "
            f"{weekly_rows_read:,} rows read | "
            f"{weekly_rows_kept:,} kept"
        )


# ------------------------------------------------------------
# 5. Combine only filtered weekly rows
# ------------------------------------------------------------

if not weekly_parts:

    raise ValueError(
        "No weekly rows were retained. "
        "Check Week_Start parsing and change-date range."
    )


weekly = pd.concat(
    weekly_parts,
    ignore_index=True
)

del weekly_parts
gc.collect()


# ------------------------------------------------------------
# 6. Final type cleanup
# ------------------------------------------------------------

weekly["Week_Start"] = pd.to_datetime(
    weekly["Week_Start"],
    errors="coerce"
)

strict["VRLOCATION"] = (
    strict["VRLOCATION"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

strict["LANE"] = (
    strict["LANE"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

strict["price_tier"] = (
    strict["price_tier"]
    .astype("string")
    .fillna("")
    .str.strip()
)


# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MEMORY-SAFE LOAD COMPLETE")
print("=" * 80)

print(
    "Weekly rows read:",
    f"{weekly_rows_read:,}"
)

print(
    "Weekly rows retained:",
    f"{len(weekly):,}"
)

print(
    "Weekly unique locations:",
    weekly["VRLOCATION"].nunique()
)

print(
    "Weekly date range:",
    weekly["Week_Start"].min(),
    "to",
    weekly["Week_Start"].max()
)

print(
    "\nStrict events:",
    f"{len(strict):,}"
)

print(
    "Strict locations:",
    strict["VRLOCATION"].nunique()
)

print(
    "Strict corporate change dates:",
    strict["Corporate_Change_ID"].nunique()
)

print(
    "\nRPU-ready events:",
    f"{len(rpu_ready):,}"
)

LOADING V6.1 DATA - MEMORY SAFE
Events loaded : 9,251
Strict loaded : 1,843
RPU loaded    : 504

Weekly panel date range needed:
From: 2022-12-04 00:00:00
To  : 2026-04-19 00:00:00

Strict treated segments: 1,335


MemoryError: Unable to allocate 18.2 MiB for an array with shape (18, 132742) and data type float64

## 2. Helper functions

In [ ]:

SEGMENT_KEYS = ["VRLOCATION", "LANE", "price_tier"]

def safe_div(a, b):
    if pd.isna(b) or b == 0:
        return np.nan
    return a / b

def safe_pct_change(post, pre):
    if pd.isna(pre) or pre == 0 or pd.isna(post):
        return np.nan
    return (post - pre) / pre

def linear_slope(values):
    vals = np.asarray(values, dtype=float)
    mask = np.isfinite(vals)

    if mask.sum() < 2:
        return np.nan

    x = np.arange(len(vals))[mask]
    y = vals[mask]

    return np.polyfit(x, y, 1)[0]

def coeff_var(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) < 2:
        return np.nan

    mean = vals.mean()

    if mean == 0:
        return np.nan

    return vals.std(ddof=1) / abs(mean)

def summarize_segment_window(df):
    if df.empty:
        return None

    reg = df["Registered_Units"].sum()
    sold = df["Sold_Units"].sum()

    fee_rev = df["BuyFee_Revenue"].sum()

    def wmean(sum_col, count_col):
        if sum_col not in df.columns or count_col not in df.columns:
            return np.nan

        den = df[count_col].sum()

        if den <= 0:
            return np.nan

        return df[sum_col].sum() / den

    return {
        "observed_weeks": df["Week_Start"].nunique(),
        "registered": reg,
        "sold": sold,
        "conversion": sold / reg if reg > 0 else np.nan,
        "rpu": fee_rev / sold if sold > 0 else np.nan,
        "buyfee_coverage": (
            df["Sold_With_BuyFee"].sum() / sold
            if sold > 0
            else np.nan
        ),
        "avg_saleamt": wmean("SaleAmt_Final_Sum", "SaleAmt_Final_Count"),
        "avg_age": wmean("Vehicle_Age_Sum", "Vehicle_Age_Count"),
        "avg_mileage": wmean("Mileage_Sum", "Mileage_Count"),
        "avg_condition": wmean("Condition_Sum", "Condition_Count"),
        "avg_high_bid": wmean("High_Bid_Sum", "High_Bid_Count"),
        "avg_floor": wmean("Floor_Sum", "Floor_Count"),
        "volume_trend": linear_slope(df["Sold_Units"].values),
        "conversion_trend": linear_slope(df["Conversion"].values),
        "volume_cv": coeff_var(df["Sold_Units"].values),
        "conversion_cv": coeff_var(df["Conversion"].values),
    }

def event_week_from_date(dt):
    dt = pd.Timestamp(dt)
    return dt - pd.to_timedelta(dt.weekday(), unit="D")



## 3. Build event-date snapshots for all possible treated/control segments

For each corporate fee-change date, this cell summarizes every segment that has enough data in the same 8-week pre/post calendar window.

This makes control matching much faster than scanning the full weekly panel separately for every treated event.


In [ ]:

unique_change_dates = (
    strict[
        ["Corporate_Change_ID", "BUYFEE_CHANGE_DATE"]
    ]
    .drop_duplicates()
    .sort_values("BUYFEE_CHANGE_DATE")
)

snapshot_parts = []

for row_num, row in unique_change_dates.reset_index(drop=True).iterrows():

    corp_id = row["Corporate_Change_ID"]
    change_date = pd.Timestamp(row["BUYFEE_CHANGE_DATE"])
    event_week = event_week_from_date(change_date)

    pre_start = event_week - pd.Timedelta(weeks=PRIMARY_WINDOW)
    pre_end = event_week - pd.Timedelta(weeks=1)

    post_start = event_week + pd.Timedelta(weeks=1)
    post_end = event_week + pd.Timedelta(weeks=PRIMARY_WINDOW)

    window = weekly[
        weekly["Week_Start"].between(pre_start, post_end)
    ].copy()

    if window.empty:
        continue

    rows = []

    for key, seg in window.groupby(SEGMENT_KEYS, sort=False):

        pre = seg[
            seg["Week_Start"].between(pre_start, pre_end)
        ]

        post = seg[
            seg["Week_Start"].between(post_start, post_end)
        ]

        a = summarize_segment_window(pre)
        b = summarize_segment_window(post)

        if a is None or b is None:
            continue

        if (
            a["observed_weeks"] < MIN_OBSERVED_WEEKS
            or b["observed_weeks"] < MIN_OBSERVED_WEEKS
            or a["registered"] < MIN_REGISTERED_PRE
            or b["registered"] < MIN_REGISTERED_POST
        ):
            continue

        loc, lane, tier = key

        rows.append({
            "Corporate_Change_ID": corp_id,
            "BUYFEE_CHANGE_DATE": change_date,
            "Event_Week": event_week,
            "VRLOCATION": loc,
            "LANE": lane,
            "price_tier": tier,

            "pre_observed_weeks": a["observed_weeks"],
            "post_observed_weeks": b["observed_weeks"],

            "pre_registered": a["registered"],
            "post_registered": b["registered"],

            "pre_sold": a["sold"],
            "post_sold": b["sold"],

            "pre_conversion": a["conversion"],
            "post_conversion": b["conversion"],

            "pre_rpu": a["rpu"],
            "post_rpu": b["rpu"],

            "pre_buyfee_coverage": a["buyfee_coverage"],
            "post_buyfee_coverage": b["buyfee_coverage"],

            "pre_avg_saleamt": a["avg_saleamt"],
            "post_avg_saleamt": b["avg_saleamt"],

            "pre_avg_age": a["avg_age"],
            "post_avg_age": b["avg_age"],

            "pre_avg_mileage": a["avg_mileage"],
            "post_avg_mileage": b["avg_mileage"],

            "pre_avg_condition": a["avg_condition"],
            "post_avg_condition": b["avg_condition"],

            "pre_avg_high_bid": a["avg_high_bid"],
            "post_avg_high_bid": b["avg_high_bid"],

            "pre_avg_floor": a["avg_floor"],
            "post_avg_floor": b["avg_floor"],

            "pre_volume_trend": a["volume_trend"],
            "pre_conversion_trend": a["conversion_trend"],

            "pre_volume_cv": a["volume_cv"],
            "pre_conversion_cv": a["conversion_cv"],
        })

    if rows:
        snapshot_parts.append(pd.DataFrame(rows))

    if (row_num + 1) % 5 == 0:
        print(
            f"Processed {row_num + 1:,} / "
            f"{len(unique_change_dates):,} corporate change dates"
        )

snapshots = pd.concat(
    snapshot_parts,
    ignore_index=True
)

snapshots.to_csv(
    OUTPUT_DIR / "01_EVENT_DATE_SEGMENT_SNAPSHOTS.csv",
    index=False
)

print("Snapshot rows:", f"{len(snapshots):,}")
print("Change dates represented:", snapshots["Corporate_Change_ID"].nunique())


## 4. Mark treated segments and contaminated controls

In [ ]:

# Exact treated segments on each corporate date
treated_key = strict[
    [
        "Corporate_Change_ID",
        "VRLOCATION",
        "LANE",
        "price_tier",
        "Segment_Change_ID",
        "BUYFEE_CHANGE_AMOUNT",
        "PREVIOUS_BUYFEE",
        "FEE_CHANGE_PCT"
    ]
].drop_duplicates()

snapshots = snapshots.merge(
    treated_key,
    on=[
        "Corporate_Change_ID",
        "VRLOCATION",
        "LANE",
        "price_tier"
    ],
    how="left"
)

snapshots["Is_Treated"] = snapshots["Segment_Change_ID"].notna()

# Build event lookup by segment to identify nearby changes
event_dates_by_segment = {
    key: grp["BUYFEE_CHANGE_DATE"].sort_values().tolist()
    for key, grp in events.groupby(
        ["VRLOCATION", "LANE", "price_tier"],
        sort=False
    )
}

def has_nearby_change(row):
    key = (
        row["VRLOCATION"],
        row["LANE"],
        row["price_tier"]
    )

    dates = event_dates_by_segment.get(key, [])

    if not dates:
        return False

    event_date = pd.Timestamp(row["BUYFEE_CHANGE_DATE"])

    for d in dates:
        weeks = abs(
            (pd.Timestamp(d) - event_date).days
        ) / 7

        if weeks <= CONTROL_CHANGE_EXCLUSION_WEEKS:
            return True

    return False

# For treated rows nearby change is expected.
# For controls it makes them contaminated.
snapshots["Has_Nearby_Fee_Change"] = snapshots.apply(
    has_nearby_change,
    axis=1
)

snapshots["Eligible_Control"] = (
    ~snapshots["Is_Treated"]
    &
    ~snapshots["Has_Nearby_Fee_Change"]
)

snapshots.to_csv(
    OUTPUT_DIR / "02_SNAPSHOT_TREATMENT_CONTROL_FLAGS.csv",
    index=False
)

print("Treated snapshot rows:", f"{snapshots['Is_Treated'].sum():,}")
print("Eligible control snapshot rows:", f"{snapshots['Eligible_Control'].sum():,}")



## 5. Match controls

Matching is based only on **pre-treatment information**.

Features include:

- price tier / sale amount
- registered units
- sold units
- conversion
- RPU when available
- vehicle age
- mileage
- condition
- high bid
- floor
- volume trend
- conversion trend
- volatility

Controls are chosen from the **same corporate change date**, which means treated and control units experience the same macro calendar period.

By default, controls from the same location are excluded.


In [ ]:

match_features = [
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
    "pre_avg_age",
    "pre_avg_mileage",
    "pre_avg_condition",
    "pre_avg_high_bid",
    "pre_avg_floor",
    "pre_volume_trend",
    "pre_conversion_trend",
    "pre_volume_cv",
    "pre_conversion_cv",
]

match_parts = []
unmatched_rows = []

for corp_id, treated_group in snapshots[
    snapshots["Is_Treated"]
].groupby("Corporate_Change_ID"):

    candidates_all = snapshots[
        (snapshots["Corporate_Change_ID"].eq(corp_id))
        &
        snapshots["Eligible_Control"]
    ].copy()

    if candidates_all.empty:
        for _, tr in treated_group.iterrows():
            unmatched_rows.append({
                "Segment_Change_ID": tr["Segment_Change_ID"],
                "Corporate_Change_ID": corp_id,
                "Reason": "No eligible controls on event date"
            })
        continue

    for _, tr in treated_group.iterrows():

        candidates = candidates_all.copy()

        if EXCLUDE_SAME_LOCATION:
            cross_loc = candidates[
                candidates["VRLOCATION"].ne(
                    tr["VRLOCATION"]
                )
            ]

            # fallback to same-location only if no cross-location pool exists
            if len(cross_loc) >= N_CONTROLS:
                candidates = cross_loc

        # Restrict to roughly comparable price level
        tr_sale = tr["pre_avg_saleamt"]

        if pd.notna(tr_sale):
            near_price = candidates[
                candidates["pre_avg_saleamt"].between(
                    tr_sale - PRICE_TIER_TOLERANCE,
                    tr_sale + PRICE_TIER_TOLERANCE
                )
            ]

            if len(near_price) >= N_CONTROLS:
                candidates = near_price

        if len(candidates) < 1:
            unmatched_rows.append({
                "Segment_Change_ID": tr["Segment_Change_ID"],
                "Corporate_Change_ID": corp_id,
                "Reason": "No controls after restrictions"
            })
            continue

        combined = pd.concat(
            [
                pd.DataFrame([tr]),
                candidates
            ],
            ignore_index=True
        )

        X = combined[match_features].copy()

        for c in match_features:
            X[c] = pd.to_numeric(
                X[c],
                errors="coerce"
            )

            med = X[c].median()

            if pd.isna(med):
                X[c] = 0.0
            else:
                X[c] = X[c].fillna(med)

        scaler = StandardScaler()

        Xs = scaler.fit_transform(X)

        tr_vec = Xs[[0]]
        cand_vec = Xs[1:]

        n_neighbors = min(
            N_CONTROLS,
            len(candidates)
        )

        nn = NearestNeighbors(
            n_neighbors=n_neighbors,
            metric="euclidean"
        )

        nn.fit(cand_vec)

        distances, indices = nn.kneighbors(
            tr_vec
        )

        selected = candidates.iloc[
            indices[0]
        ].copy()

        selected["Match_Distance"] = distances[0]

        selected = selected[
            selected["Match_Distance"] <= MAX_MATCH_DISTANCE
        ].copy()

        if selected.empty:
            unmatched_rows.append({
                "Segment_Change_ID": tr["Segment_Change_ID"],
                "Corporate_Change_ID": corp_id,
                "Reason": "Nearest control exceeds distance threshold"
            })
            continue

        for rank, (_, ctrl) in enumerate(
            selected.iterrows(),
            start=1
        ):

            row = {
                "Segment_Change_ID": tr["Segment_Change_ID"],
                "Corporate_Change_ID": corp_id,
                "BUYFEE_CHANGE_DATE": tr["BUYFEE_CHANGE_DATE"],
                "BUYFEE_CHANGE_AMOUNT": tr["BUYFEE_CHANGE_AMOUNT"],
                "PREVIOUS_BUYFEE": tr["PREVIOUS_BUYFEE"],
                "FEE_CHANGE_PCT": tr["FEE_CHANGE_PCT"],

                "Treated_Location": tr["VRLOCATION"],
                "Treated_Lane": tr["LANE"],
                "Treated_Price_Tier": tr["price_tier"],

                "Control_Rank": rank,
                "Control_Location": ctrl["VRLOCATION"],
                "Control_Lane": ctrl["LANE"],
                "Control_Price_Tier": ctrl["price_tier"],
                "Match_Distance": ctrl["Match_Distance"],
            }

            # carry outcome and covariate values for both sides
            for c in [
                "pre_registered", "post_registered",
                "pre_sold", "post_sold",
                "pre_conversion", "post_conversion",
                "pre_rpu", "post_rpu",
                "pre_buyfee_coverage", "post_buyfee_coverage",
                "pre_avg_saleamt", "post_avg_saleamt",
                "pre_avg_age", "post_avg_age",
                "pre_avg_mileage", "post_avg_mileage",
                "pre_avg_condition", "post_avg_condition",
                "pre_avg_high_bid", "post_avg_high_bid",
                "pre_avg_floor", "post_avg_floor",
                "pre_volume_trend",
                "pre_conversion_trend",
                "pre_volume_cv",
                "pre_conversion_cv",
            ]:
                row[f"T_{c}"] = tr[c]
                row[f"C_{c}"] = ctrl[c]

            match_parts.append(row)

matched_pairs = pd.DataFrame(match_parts)
unmatched = pd.DataFrame(unmatched_rows)

matched_pairs.to_csv(
    OUTPUT_DIR / "03_MATCHED_CONTROL_PAIRS.csv",
    index=False
)

unmatched.to_csv(
    OUTPUT_DIR / "04_UNMATCHED_TREATED_EVENTS.csv",
    index=False
)

print("Matched pair rows:", f"{len(matched_pairs):,}")
print("Matched treated events:", matched_pairs["Segment_Change_ID"].nunique())
print("Strict treated events:", strict["Segment_Change_ID"].nunique())
print(
    "Match rate:",
    round(
        matched_pairs["Segment_Change_ID"].nunique()
        / strict["Segment_Change_ID"].nunique(),
        4
    )
)


## 6. Match-quality diagnostics

In [ ]:

balance_rows = []

for feature in match_features:

    t = pd.to_numeric(
        matched_pairs[f"T_{feature}"],
        errors="coerce"
    )

    c = pd.to_numeric(
        matched_pairs[f"C_{feature}"],
        errors="coerce"
    )

    pooled_sd = np.sqrt(
        (
            np.nanvar(t, ddof=1)
            +
            np.nanvar(c, ddof=1)
        ) / 2
    )

    smd = (
        (np.nanmean(t) - np.nanmean(c))
        / pooled_sd
        if pooled_sd > 0
        else np.nan
    )

    balance_rows.append({
        "Feature": feature,
        "Treated_Mean": np.nanmean(t),
        "Control_Mean": np.nanmean(c),
        "Standardized_Mean_Difference": smd,
        "Abs_SMD": abs(smd) if pd.notna(smd) else np.nan
    })

balance = pd.DataFrame(balance_rows)

balance.to_csv(
    OUTPUT_DIR / "05_MATCH_BALANCE.csv",
    index=False
)

match_quality = pd.DataFrame([
    {
        "Metric": "Strict_Treated_Events",
        "Value": strict["Segment_Change_ID"].nunique()
    },
    {
        "Metric": "Matched_Treated_Events",
        "Value": matched_pairs["Segment_Change_ID"].nunique()
    },
    {
        "Metric": "Match_Rate",
        "Value": (
            matched_pairs["Segment_Change_ID"].nunique()
            / strict["Segment_Change_ID"].nunique()
        )
    },
    {
        "Metric": "Median_Match_Distance",
        "Value": matched_pairs["Match_Distance"].median()
    },
    {
        "Metric": "Max_Abs_SMD",
        "Value": balance["Abs_SMD"].max()
    },
    {
        "Metric": "Median_Abs_SMD",
        "Value": balance["Abs_SMD"].median()
    },
    {
        "Metric": "Features_Abs_SMD_Above_0.20",
        "Value": int((balance["Abs_SMD"] > 0.20).sum())
    },
])

match_quality.to_csv(
    OUTPUT_DIR / "06_MATCH_QUALITY_SUMMARY.csv",
    index=False
)

display(match_quality)
display(balance.sort_values("Abs_SMD", ascending=False))


## 7. Calculate Difference-in-Differences outcomes

In [ ]:

# Individual control-pair effects
matched_pairs["T_Volume_Change_Pct"] = (
    matched_pairs["T_post_sold"]
    / matched_pairs["T_pre_sold"]
    - 1
)

matched_pairs["C_Volume_Change_Pct"] = (
    matched_pairs["C_post_sold"]
    / matched_pairs["C_pre_sold"]
    - 1
)

matched_pairs.loc[
    matched_pairs["T_pre_sold"].le(0),
    "T_Volume_Change_Pct"
] = np.nan

matched_pairs.loc[
    matched_pairs["C_pre_sold"].le(0),
    "C_Volume_Change_Pct"
] = np.nan

matched_pairs["DiD_Volume_Change_Pct"] = (
    matched_pairs["T_Volume_Change_Pct"]
    - matched_pairs["C_Volume_Change_Pct"]
)

matched_pairs["T_Conversion_Change_pp"] = (
    matched_pairs["T_post_conversion"]
    - matched_pairs["T_pre_conversion"]
) * 100

matched_pairs["C_Conversion_Change_pp"] = (
    matched_pairs["C_post_conversion"]
    - matched_pairs["C_pre_conversion"]
) * 100

matched_pairs["DiD_Conversion_Change_pp"] = (
    matched_pairs["T_Conversion_Change_pp"]
    - matched_pairs["C_Conversion_Change_pp"]
)

matched_pairs["T_RPU_Change_Dollar"] = (
    matched_pairs["T_post_rpu"]
    - matched_pairs["T_pre_rpu"]
)

matched_pairs["C_RPU_Change_Dollar"] = (
    matched_pairs["C_post_rpu"]
    - matched_pairs["C_pre_rpu"]
)

matched_pairs["DiD_RPU_Change_Dollar"] = (
    matched_pairs["T_RPU_Change_Dollar"]
    - matched_pairs["C_RPU_Change_Dollar"]
)

matched_pairs["DiD_Fee_Pass_Through"] = (
    matched_pairs["DiD_RPU_Change_Dollar"]
    / matched_pairs["BUYFEE_CHANGE_AMOUNT"]
)

# Aggregate multiple controls to one row per treated event
agg = (
    matched_pairs.groupby(
        [
            "Segment_Change_ID",
            "Corporate_Change_ID",
            "BUYFEE_CHANGE_DATE",
            "BUYFEE_CHANGE_AMOUNT",
            "PREVIOUS_BUYFEE",
            "FEE_CHANGE_PCT",
            "Treated_Location",
            "Treated_Lane",
            "Treated_Price_Tier"
        ],
        as_index=False
    )
    .agg(
        Number_of_Controls=("Control_Rank", "count"),
        Median_Match_Distance=("Match_Distance", "median"),

        Treated_Volume_Change_Pct=("T_Volume_Change_Pct", "first"),
        Control_Volume_Change_Pct=("C_Volume_Change_Pct", "mean"),
        DiD_Volume_Change_Pct=("DiD_Volume_Change_Pct", "mean"),

        Treated_Conversion_Change_pp=("T_Conversion_Change_pp", "first"),
        Control_Conversion_Change_pp=("C_Conversion_Change_pp", "mean"),
        DiD_Conversion_Change_pp=("DiD_Conversion_Change_pp", "mean"),

        Treated_RPU_Change_Dollar=("T_RPU_Change_Dollar", "first"),
        Control_RPU_Change_Dollar=("C_RPU_Change_Dollar", "mean"),
        DiD_RPU_Change_Dollar=("DiD_RPU_Change_Dollar", "mean"),

        DiD_Fee_Pass_Through=("DiD_Fee_Pass_Through", "mean"),

        pre_registered=("T_pre_registered", "first"),
        pre_sold=("T_pre_sold", "first"),
        pre_conversion=("T_pre_conversion", "first"),
        pre_rpu=("T_pre_rpu", "first"),
        pre_avg_saleamt=("T_pre_avg_saleamt", "first"),
        pre_avg_age=("T_pre_avg_age", "first"),
        pre_avg_mileage=("T_pre_avg_mileage", "first"),
        pre_avg_condition=("T_pre_avg_condition", "first"),
        pre_avg_high_bid=("T_pre_avg_high_bid", "first"),
        pre_avg_floor=("T_pre_avg_floor", "first"),
        pre_volume_trend=("T_pre_volume_trend", "first"),
        pre_conversion_trend=("T_pre_conversion_trend", "first"),
        pre_volume_cv=("T_pre_volume_cv", "first"),
        pre_conversion_cv=("T_pre_conversion_cv", "first"),
    )
)

agg.to_csv(
    OUTPUT_DIR / "07_DID_EVENT_LEVEL_RESULTS.csv",
    index=False
)

print("Matched treated events:", f"{len(agg):,}")
display(agg.head())


## 8. DiD fixed-effects regressions

In [ ]:

did_fe_rows = []

targets = {
    "Volume": "DiD_Volume_Change_Pct",
    "Conversion": "DiD_Conversion_Change_pp",
    "RPU": "DiD_RPU_Change_Dollar",
}

for name, target in targets.items():

    d = agg.copy()

    numeric_controls = [
        "BUYFEE_CHANGE_AMOUNT",
        "FEE_CHANGE_PCT",
        "PREVIOUS_BUYFEE",
        "pre_registered",
        "pre_sold",
        "pre_conversion",
        "pre_rpu",
        "pre_avg_saleamt",
        "pre_avg_age",
        "pre_avg_mileage",
        "pre_avg_condition",
        "pre_avg_high_bid",
        "pre_avg_floor",
        "pre_volume_trend",
        "pre_conversion_trend",
        "pre_volume_cv",
        "pre_conversion_cv",
        target,
    ]

    for c in numeric_controls:
        if c in d.columns:
            d[c] = pd.to_numeric(
                d[c],
                errors="coerce"
            )

    controls = [
        c for c in numeric_controls
        if c != target
        and c in d.columns
        and d[c].notna().sum() >= max(30, int(len(d) * 0.25))
    ]

    for c in controls:
        d[c] = d[c].fillna(d[c].median())

    d = d.dropna(
        subset=[
            target,
            "BUYFEE_CHANGE_AMOUNT",
            "Corporate_Change_ID",
            "Treated_Location"
        ]
    ).copy()

    if len(d) < 50:
        did_fe_rows.append({
            "Target": name,
            "Rows": len(d),
            "Status": "INSUFFICIENT_ROWS"
        })
        continue

    formula = (
        f"{target} ~ "
        + " + ".join(controls)
        + " + C(Treated_Location)"
        + " + C(Corporate_Change_ID)"
    )

    try:
        model = smf.ols(
            formula=formula,
            data=d
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": d["Corporate_Change_ID"]
            }
        )

        clustered = True

    except Exception:
        model = smf.ols(
            formula=formula,
            data=d
        ).fit(
            cov_type="HC3"
        )

        clustered = False

    coef = model.params.get(
        "BUYFEE_CHANGE_AMOUNT",
        np.nan
    )

    pval = model.pvalues.get(
        "BUYFEE_CHANGE_AMOUNT",
        np.nan
    )

    if "BUYFEE_CHANGE_AMOUNT" in model.params.index:
        ci = model.conf_int().loc[
            "BUYFEE_CHANGE_AMOUNT"
        ].tolist()
    else:
        ci = [np.nan, np.nan]

    did_fe_rows.append({
        "Target": name,
        "Rows": len(d),
        "Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),
        "R2": model.rsquared,
        "Adj_R2": model.rsquared_adj,
        "Fee_Dollar_Coefficient": coef,
        "Fee_Dollar_PValue": pval,
        "Fee_CI_Low": ci[0],
        "Fee_CI_High": ci[1],
        "Clustered_SE": clustered,
        "Status": "OK",
    })

did_fixed_effects = pd.DataFrame(did_fe_rows)

did_fixed_effects.to_csv(
    OUTPUT_DIR / "08_DID_FIXED_EFFECTS_SUMMARY.csv",
    index=False
)

display(did_fixed_effects)


## 9. Grouped-by-Corporate_Change_ID stability

In [ ]:

stability_rows = []
loo_rows = []

for name, target in targets.items():

    d = agg.dropna(
        subset=[
            target,
            "BUYFEE_CHANGE_AMOUNT",
            "Corporate_Change_ID"
        ]
    ).copy()

    groups = d["Corporate_Change_ID"].unique().tolist()

    coefs = []

    for held_out in groups:

        train = d[
            d["Corporate_Change_ID"].ne(held_out)
        ].copy()

        if (
            len(train) < 100
            or train["Corporate_Change_ID"].nunique() < 5
        ):
            continue

        try:
            model = smf.ols(
                formula=(
                    f"{target} ~ BUYFEE_CHANGE_AMOUNT"
                    " + FEE_CHANGE_PCT"
                    " + pre_registered"
                    " + pre_sold"
                    " + pre_conversion"
                    " + pre_avg_saleamt"
                    " + C(Treated_Location)"
                ),
                data=train
            ).fit(cov_type="HC3")

            coef = model.params.get(
                "BUYFEE_CHANGE_AMOUNT",
                np.nan
            )

            coefs.append(coef)

            loo_rows.append({
                "Target": name,
                "Held_Out_Corporate_Change_ID": held_out,
                "Fee_Dollar_Coefficient": coef,
                "Rows": len(train),
                "Train_Change_Dates": train["Corporate_Change_ID"].nunique()
            })

        except Exception:
            continue

    coefs = pd.Series(coefs).dropna()

    if len(coefs):

        median_coef = coefs.median()

        sign_stability = (
            np.sign(coefs)
            == np.sign(median_coef)
        ).mean()

        stability_rows.append({
            "Target": name,
            "LOO_Runs": len(coefs),
            "Median_Fee_Coefficient": median_coef,
            "Coefficient_Min": coefs.min(),
            "Coefficient_Max": coefs.max(),
            "Sign_Stability": sign_stability,
        })

stability = pd.DataFrame(stability_rows)
loo_detail = pd.DataFrame(loo_rows)

stability.to_csv(
    OUTPUT_DIR / "09_DID_COEFFICIENT_STABILITY.csv",
    index=False
)

loo_detail.to_csv(
    OUTPUT_DIR / "10_DID_LOO_COEFFICIENTS.csv",
    index=False
)

display(stability)


## 10. Scenario evidence after control adjustment

In [ ]:

scenario_rows = []

for fee in SCENARIOS:

    d = agg[
        agg["BUYFEE_CHANGE_AMOUNT"].between(
            fee - 2.5,
            fee + 2.5
        )
    ].copy()

    scenario_rows.append({
        "Fee_Increase": fee,
        "Matched_Events": len(d),
        "Unique_Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),

        "Median_DiD_Volume_Change_Pct": d["DiD_Volume_Change_Pct"].median(),
        "Mean_DiD_Volume_Change_Pct": d["DiD_Volume_Change_Pct"].mean(),

        "Median_DiD_Conversion_Change_pp": d["DiD_Conversion_Change_pp"].median(),
        "Mean_DiD_Conversion_Change_pp": d["DiD_Conversion_Change_pp"].mean(),

        "Median_DiD_RPU_Change_Dollar": d["DiD_RPU_Change_Dollar"].median(),
        "Mean_DiD_RPU_Change_Dollar": d["DiD_RPU_Change_Dollar"].mean(),

        "Median_DiD_Fee_Pass_Through": d["DiD_Fee_Pass_Through"].median(),
    })

scenario_did = pd.DataFrame(scenario_rows)

scenario_did.to_csv(
    OUTPUT_DIR / "11_DID_SCENARIO_EVIDENCE.csv",
    index=False
)

display(scenario_did)


## 11. Distribution diagnostics

In [ ]:

distribution_rows = []

for name, target in targets.items():

    s = pd.to_numeric(
        agg[target],
        errors="coerce"
    ).dropna()

    distribution_rows.append({
        "Target": name,
        "N": len(s),
        "Mean": s.mean(),
        "Median": s.median(),
        "P10": s.quantile(0.10),
        "P25": s.quantile(0.25),
        "P75": s.quantile(0.75),
        "P90": s.quantile(0.90),
        "Pct_Negative": (s < 0).mean(),
        "Pct_Positive": (s > 0).mean(),
    })

did_distribution = pd.DataFrame(distribution_rows)

did_distribution.to_csv(
    OUTPUT_DIR / "12_DID_OUTCOME_DISTRIBUTIONS.csv",
    index=False
)

display(did_distribution)


## 12. Simple DiD scenario charts

In [ ]:

def scenario_plot(y_col, title, ylabel, filename):

    d = scenario_did.copy()

    plt.figure(figsize=(8, 5))

    plt.plot(
        d["Fee_Increase"],
        d[y_col],
        marker="o"
    )

    plt.axhline(0, linestyle=":")

    plt.title(title)
    plt.xlabel("Buy Fee increase ($)")
    plt.ylabel(ylabel)

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / filename,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

scenario_plot(
    "Median_DiD_Volume_Change_Pct",
    "Control-adjusted volume response",
    "Median DiD volume change",
    "13_DID_VOLUME_SCENARIOS.png"
)

scenario_plot(
    "Median_DiD_Conversion_Change_pp",
    "Control-adjusted conversion response",
    "Median DiD conversion change (pp)",
    "14_DID_CONVERSION_SCENARIOS.png"
)

scenario_plot(
    "Median_DiD_RPU_Change_Dollar",
    "Control-adjusted Buy Fee RPU response",
    "Median DiD RPU change ($)",
    "15_DID_RPU_SCENARIOS.png"
)


## 13. V8 readiness assessment

In [ ]:

match_rate = (
    matched_pairs["Segment_Change_ID"].nunique()
    / strict["Segment_Change_ID"].nunique()
)

readiness_rows = []

for name, target in targets.items():

    d = agg.dropna(subset=[target]).copy()

    locations = d["Treated_Location"].nunique()
    dates = d["Corporate_Change_ID"].nunique()

    stab_row = stability[
        stability["Target"].eq(name)
    ]

    sign_stability = (
        stab_row.iloc[0]["Sign_Stability"]
        if not stab_row.empty
        else np.nan
    )

    fe_row = did_fixed_effects[
        did_fixed_effects["Target"].eq(name)
    ]

    fee_coef = (
        fe_row.iloc[0]["Fee_Dollar_Coefficient"]
        if not fe_row.empty
        and "Fee_Dollar_Coefficient" in fe_row.columns
        else np.nan
    )

    fee_p = (
        fe_row.iloc[0]["Fee_Dollar_PValue"]
        if not fe_row.empty
        and "Fee_Dollar_PValue" in fe_row.columns
        else np.nan
    )

    reasons = []

    if (
        len(d) < 100
        or dates < 5
        or locations < 5
    ):
        status = "IMPROVE DATA"
        reasons.append(
            "Insufficient matched independent support."
        )

    else:

        go = (
            match_rate >= MIN_MATCH_RATE_GO
            and dates >= MIN_UNIQUE_DATES_GO
            and locations >= MIN_UNIQUE_LOCATIONS_GO
            and pd.notna(sign_stability)
            and sign_stability >= MIN_SIGN_STABILITY_GO
        )

        if go:
            status = "GO"
            reasons.append(
                "Matched-control support and coefficient stability meet V8 gates."
            )
        else:
            status = "RESEARCH ONLY"

            if match_rate < MIN_MATCH_RATE_GO:
                reasons.append(
                    "Matched-control coverage below GO threshold."
                )

            if dates < MIN_UNIQUE_DATES_GO:
                reasons.append(
                    "Independent corporate change-date support below GO threshold."
                )

            if locations < MIN_UNIQUE_LOCATIONS_GO:
                reasons.append(
                    "Location support below GO threshold."
                )

            if (
                pd.isna(sign_stability)
                or sign_stability < MIN_SIGN_STABILITY_GO
            ):
                reasons.append(
                    "Fee-effect sign is unstable across held-out change dates."
                )

    readiness_rows.append({
        "Target": name,
        "Readiness": status,
        "Matched_Events": len(d),
        "Unique_Locations": locations,
        "Corporate_Change_Dates": dates,
        "Overall_Match_Rate": match_rate,
        "Fee_Dollar_Coefficient": fee_coef,
        "Fee_Dollar_PValue": fee_p,
        "Fee_Coefficient_Sign_Stability": sign_stability,
        "Reason": " | ".join(reasons)
    })

readiness = pd.DataFrame(readiness_rows)

readiness.to_csv(
    OUTPUT_DIR / "16_V8_READINESS_ASSESSMENT.csv",
    index=False
)

display(readiness)


## 14. Executive summary

In [ ]:

executive = readiness.merge(
    did_fixed_effects[
        [
            "Target",
            "R2",
            "Adj_R2"
        ]
    ],
    on="Target",
    how="left"
)

executive = executive.merge(
    did_distribution[
        [
            "Target",
            "Median",
            "Pct_Negative",
            "Pct_Positive"
        ]
    ],
    on="Target",
    how="left"
)

executive = executive.rename(
    columns={
        "R2": "DiD_FE_R2",
        "Adj_R2": "DiD_FE_Adj_R2",
        "Median": "Median_Control_Adjusted_Effect"
    }
)

executive.to_csv(
    OUTPUT_DIR / "17_V8_EXECUTIVE_SUMMARY.csv",
    index=False
)

display(executive)



## Files to share back

Please send:

1. `17_V8_EXECUTIVE_SUMMARY.csv`
2. `16_V8_READINESS_ASSESSMENT.csv`
3. `11_DID_SCENARIO_EVIDENCE.csv`
4. `08_DID_FIXED_EFFECTS_SUMMARY.csv`
5. `09_DID_COEFFICIENT_STABILITY.csv`
6. `06_MATCH_QUALITY_SUMMARY.csv`
7. `05_MATCH_BALANCE.csv`
8. `12_DID_OUTCOME_DISTRIBUTIONS.csv`

The three scenario PNGs are also helpful.

## What comes next

If V8 shows that RPU and/or Conversion remain stable after matched-control adjustment, V9 can build the final decision engine:

- +$10 / +$15 / +$20 / +$25 / +$30 scenarios
- expected RPU gain
- conversion-risk probability
- volume guardrail
- expected revenue impact
- confidence / evidence strength
- location × lane × price-tier recommendations
